# XGBoost Final Focused Ablation: Remove 5 Highest-Priority XGBoost Hotspots

This notebook removes the final five XGBoost-only hotspot candidates together, retrains the same frozen by-gene+mRNA XGBoost setup, and re-analyzes the outcome.

In [1]:
%load_ext autoreload
%autoreload 2

import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import seaborn as sns
from scipy.stats import pearsonr, spearmanr
from sklearn.metrics import mean_absolute_error, mean_squared_error
from xgboost import XGBRegressor

project_root = Path.cwd().resolve()
while not (project_root / "utils").exists() and project_root != project_root.parent:
    project_root = project_root.parent

if not (project_root / "utils").exists():
    raise RuntimeError("Could not locate project root containing the utils package")

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from utils.merge_historic_data import load_merged_dataset
from utils.pipeline import SiRNADataPipeline
from utils.splitter import GroupKFoldLeakPerGroup

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)


## Build Current Pipeline Data

This ablation uses the current shared preprocessing setup: `strict_cleaning=True`, `add_mrna=True`, and `use_normalized_conditions=False`. It keeps the frozen XGBoost by-gene+mRNA Optuna hyperparameters so the only intended change is the focused row removal.

In [2]:
cmsirna_path = os.environ.get("CMSIRNA_RAW_DATA_PATH")
historic_path = os.environ.get("CMSIRNA_RAW_HISTORIC_DATA_PATH")

assert cmsirna_path, "CMSIRNA_RAW_DATA_PATH is not set"
assert historic_path, "CMSIRNA_RAW_HISTORIC_DATA_PATH is not set"

raw_df = load_merged_dataset(cmsirna_path, historic_path)
pipeline = SiRNADataPipeline(target_len=25, fetch_missing_mrna=True)
enriched_df = pipeline.enrich_dataset_with_encodings(
    raw_df,
    strict_cleaning=True,
    add_mrna=True,
)

enriched_df["patent_group"] = enriched_df.get(
    "patent_ID", pd.Series(index=enriched_df.index, dtype=object)
).fillna("HISTORIC_OR_UNKNOWN")
enriched_df["Authorization_status"] = enriched_df.get(
    "Authorization_status", pd.Series(index=enriched_df.index, dtype=object)
).fillna("UNKNOWN")
enriched_df["Title"] = enriched_df.get(
    "Title", pd.Series(index=enriched_df.index, dtype=object)
).fillna("HISTORIC_OR_UNKNOWN")


loaded 3515 historic rows
merged 43153 CMsiRNA and 3515 historic rows into 46668
Running qc and data cleaning
dropped 4233 rows for in-vivo readings
dropped 565 rows for mM readings
dropped 115 rows for unknown or unwanted cell lines
dropped 47 rows for out-of-range inhibition
dropped 1749 rows for missing or unknown concentration
dropped 796 rows for concentration > 200 nM
filled 7522 rows for missing time of administration
dropped 2198 rows with a missing or >25 nt strand
dropped 6 columns: ['Modification_locations_Sense_strand', 'Modification_locations_Antisense_strand', 'Modifications_sense_strand', 'Modifications_AntiSense_strand_3_5', 'position_Antisense_strand', 'position_Sense_strand']
Mapping mRNA structural profiles
Error reading reference dataset: [Errno 2] No such file or directory: '/home/larsena8/software/fennec/src/fennec/support_files/train_data_v1.1.0_N=27742.csv'
Loaded 47 gene sequences from local cache
Loaded 0 gene sequences from reference CSV
Building gene -> mRNA

## What This Notebook Removes

Removed hotspots:
- `APP @ 20 nM`
- `INHBE @ 100 nM`
- `PCSK9 @ 100 nM`
- `US20220380773A1 @ 20 nM`
- `Primary mouse hepatocytes`

Why: this is the final XGBoost-only focused ablation set. It combines the strongest target-specific hotspot ablation with the strongest patent-specific hotspot and the clearest recurrent problematic cell context.

Exact investigation rows that motivated these removals:
- `APP @ 20 nM`: `n_samples = 168`, `spearman = -0.556349`, `mae = 38.583055`, `mean_true = 41.410714`, `mean_pred = 57.806583`
- `INHBE @ 100 nM`: `n_samples = 180`, `spearman = -0.440447`, `mae = 37.574473`, `mean_true = 62.262167`, `mean_pred = 39.903522`
- `PCSK9 @ 100 nM`: `n_samples = 293`, `spearman = -0.212300`, `mae = 31.238838`, `mean_true = 54.427304`, `mean_pred = 54.176559`
- `US20220380773A1 @ 20 nM`: `n_samples = 168`, `spearman = -0.556349`, `mae = 38.583055`, `mean_true = 41.410714`, `mean_pred = 57.806583`
- `Primary mouse hepatocytes`: `n_samples = 306`, `spearman = -0.176535`, `mae = 58.485055`, `mean_true = 23.606536`, `mean_pred = 44.061111`

The goal is to test whether removing this small but high-confidence hotspot set produces a cleaner by-gene XGBoost signal before deep learning.


In [3]:
combo_mask = (
    ((enriched_df["gene_target_symbol_name"] == "APP") & (np.isclose(enriched_df["Concentration_nM"], 20.0, equal_nan=False)))
    | ((enriched_df["gene_target_symbol_name"] == "INHBE") & (np.isclose(enriched_df["Concentration_nM"], 100.0, equal_nan=False)))
    | ((enriched_df["gene_target_symbol_name"] == "PCSK9") & (np.isclose(enriched_df["Concentration_nM"], 100.0, equal_nan=False)))
    | ((enriched_df["patent_group"] == "US20220380773A1") & (np.isclose(enriched_df["Concentration_nM"], 20.0, equal_nan=False)))
    | (enriched_df["Cell_Type"] == "Primary mouse hepatocytes")
)
removed_df = enriched_df.loc[combo_mask].copy()
filtered_df = enriched_df.loc[~combo_mask].reset_index(drop=True)

removal_summary = pd.DataFrame([{
    "starting_rows": int(len(enriched_df)),
    "rows_removed": int(len(removed_df)),
    "rows_remaining": int(len(filtered_df)),
    "pct_removed": float(len(removed_df) / len(enriched_df)),
    "removed_hotspots": "APP@20, INHBE@100, PCSK9@100, US20220380773A1@20, Primary mouse hepatocytes",
}])

X, groups, y = pipeline.prepare_for_classical_ml(
    filtered_df,
    target_column="Inhibition",
    use_normalized_conditions=False,
)
mask = ~np.isnan(y)
X = X[mask]
groups = groups[mask]
y = y[mask]
analysis_df = filtered_df.loc[mask].reset_index(drop=True).copy()

print("Filtered dataframe shape:", analysis_df.shape)
print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)
print("Unique genes:", len(np.unique(groups)))
removal_summary


Feature matrix X shape: (34483, 1425), target y shape: (34483,)
Filtered dataframe shape: (34483, 40)
Feature matrix shape: (34483, 1425)
Target shape: (34483,)
Unique genes: 54


,starting_rows,rows_removed,rows_remaining,pct_removed,removed_hotspots
0,35444,961,34483,0.027113,"APP@20, INHBE@100, PCSK9@100, US20220380773A1@..."


## Helper Functions

In [4]:
def make_model(frozen_params):
    return XGBRegressor(
        tree_method="hist",
        n_jobs=-1,
        random_state=42,
        **frozen_params,
    )


def evaluate(y_true, y_pred):
    return {
        "spearman": float(spearmanr(y_true, y_pred).statistic),
        "pearson": float(pearsonr(y_true, y_pred).statistic),
        "rmse": float(mean_squared_error(y_true, y_pred) ** 0.5),
        "mae": float(mean_absolute_error(y_true, y_pred)),
    }


def grouped_spearman(df, group_cols, min_samples=10):
    rows = []
    for keys, group_df in df.groupby(group_cols, dropna=False):
        if len(group_df) < min_samples:
            continue
        corr = spearmanr(group_df["y_true"], group_df["y_pred"], nan_policy="omit").statistic
        if pd.isna(corr):
            continue
        if not isinstance(keys, tuple):
            keys = (keys,)
        row = {col: value for col, value in zip(group_cols, keys)}
        row.update({
            "n_samples": len(group_df),
            "spearman": float(corr),
            "mae": float(group_df["abs_error"].mean()),
            "mean_true": float(group_df["y_true"].mean()),
            "mean_pred": float(group_df["y_pred"].mean()),
        })
        rows.append(row)
    return pd.DataFrame(rows).sort_values(["spearman", "mae"], ascending=[True, False]).reset_index(drop=True)


def issue_summary(df, group_col, min_samples=20):
    rows = []
    for value, group_df in df.groupby(group_col, dropna=False):
        if len(group_df) < min_samples:
            continue
        corr = spearmanr(group_df["y_true"], group_df["y_pred"], nan_policy="omit").statistic
        rows.append({
            group_col: value,
            "n_samples": len(group_df),
            "mae": float(group_df["abs_error"].mean()),
            "rmse": float(np.sqrt(np.mean(np.square(group_df["residual"])))),
            "spearman": float(corr) if not pd.isna(corr) else np.nan,
            "mean_true": float(group_df["y_true"].mean()),
            "mean_pred": float(group_df["y_pred"].mean()),
        })
    return pd.DataFrame(rows).sort_values(["spearman", "mae"], ascending=[True, False]).reset_index(drop=True)


## Refit And Re-Analyze

The model is retrained from scratch after the focused hotspot removal, then evaluated again with the same by-gene out-of-fold procedure used in the main XGBoost investigation notebook.

In [5]:
frozen_params = {'n_estimators': 800, 'max_depth': 4, 'learning_rate': 0.15881823130907038, 'subsample': 0.8812898741586134, 'colsample_bytree': 0.7824379872752019, 'min_child_weight': 4, 'reg_lambda': 0.8342807691178866, 'reg_alpha': 1.4296995092035882, 'gamma': 0.07531958697602548}
baseline_metrics = pd.DataFrame([{'spearman': 0.45886, 'pearson': 0.453422, 'rmse': 31.421689, 'mae': 24.568816}])
gene_cv = GroupKFoldLeakPerGroup(n_splits=3, leak_n=30, random_state=42)

fold_rows = []
oof_frames = []
oof_true = []
oof_pred = []

for fold_id, (train_idx, test_idx) in enumerate(gene_cv.split(X, y, groups), start=1):
    model = make_model(frozen_params)
    model.fit(X[train_idx], y[train_idx])
    fold_pred = model.predict(X[test_idx])

    oof_true.append(y[test_idx])
    oof_pred.append(fold_pred)

    fold_frame = analysis_df.iloc[test_idx].reset_index().rename(columns={"index": "source_index"}).copy()
    fold_frame["row_index"] = test_idx
    fold_frame["fold_id"] = fold_id
    fold_frame["group"] = groups[test_idx]
    fold_frame["y_true"] = y[test_idx]
    fold_frame["y_pred"] = fold_pred
    fold_frame["residual"] = fold_frame["y_true"] - fold_frame["y_pred"]
    fold_frame["abs_error"] = fold_frame["residual"].abs()
    oof_frames.append(fold_frame)

    fold_rows.append({
        "fold_id": fold_id,
        "n_train": int(len(train_idx)),
        "n_test": int(len(test_idx)),
        "n_train_groups": int(len(np.unique(groups[train_idx]))),
        "n_test_groups": int(len(np.unique(groups[test_idx]))),
    })

predictions_df = pd.concat(oof_frames, ignore_index=True)
metrics_df = pd.DataFrame([evaluate(np.concatenate(oof_true), np.concatenate(oof_pred))])
fold_summary = pd.DataFrame(fold_rows)
comparison = pd.concat([baseline_metrics.assign(run="baseline_by_gene_mrna"), metrics_df.assign(run="ablation")], ignore_index=True)
delta_vs_baseline = metrics_df.iloc[0] - baseline_metrics.iloc[0]
delta_df = pd.DataFrame({"metric": delta_vs_baseline.index, "delta_vs_baseline": delta_vs_baseline.values})
fold_summary


,fold_id,n_train,n_test,n_train_groups,n_test_groups
0,1,23479,11004,54,16
1,2,23478,11005,54,15
2,3,23523,10960,54,17


## Fold Summary

## Baseline Comparison

In [6]:
comparison

,spearman,pearson,rmse,mae,run
0,0.458860,0.453422,31.421689,24.568816,baseline_by_gene_mrna
1,0.493612,0.491417,30.431602,23.661806,ablation


## Metric Delta Vs Baseline

In [7]:
delta_df

,metric,delta_vs_baseline
0,spearman,0.034752
1,pearson,0.037995
2,rmse,-0.990087
3,mae,-0.907010


## Overall Metrics

In [8]:
metrics_df

,spearman,pearson,rmse,mae
0,0.493612,0.491417,30.431602,23.661806


## Concentration Ranking Slice

These tables show where XGBoost preserves ranking poorly or well across concentration alone and the key concentration combinations after the final focused ablation.

In [9]:
spearman_by_concentration = grouped_spearman(predictions_df, ["Concentration_nM"], min_samples=20)
spearman_by_concentration_gene = grouped_spearman(predictions_df, ["Concentration_nM", "group"], min_samples=10)
spearman_by_concentration_patent = grouped_spearman(predictions_df, ["Concentration_nM", "patent_group"], min_samples=10)

spearman_by_concentration.sort_values(["spearman", "mae"], ascending=[True, False]).head(20)

,Concentration_nM,n_samples,spearman,mae,mean_true,mean_pred
0,15.00000,35,-0.725547,18.390196,86.038000,67.772148
1,3.00000,35,-0.639757,12.128213,71.316857,64.966675
2,150.00000,88,-0.053900,30.380491,37.002500,63.213364
3,6.66670,48,-0.011808,16.651001,78.221042,74.699287
4,0.00030,40,-0.004128,11.501416,5.699000,-0.832109
5,0.20000,120,0.017644,38.182021,18.021167,10.813755
6,0.00300,64,0.032169,16.270998,4.242813,-3.048302
7,8.00000,44,0.053700,18.810726,25.563636,42.355930
8,0.00320,111,0.079968,14.333037,1.807117,7.212139
9,0.00100,66,0.102736,17.804429,7.866212,-3.533337


## Concentration + Gene Hotspots

In [10]:
spearman_by_concentration_gene.sort_values(["spearman", "mae"], ascending=[True, False]).head(20)

,Concentration_nM,group,n_samples,spearman,mae,mean_true,mean_pred
0,15.00000,HSD17B13,35,-0.725547,18.390196,86.038000,67.772148
1,3.00000,HSD17B13,35,-0.639757,12.128213,71.316857,64.966675
2,0.24700,HSD17B13,16,-0.459161,34.587832,78.275000,49.947678
3,5.00000,HSD17B13,51,-0.449398,42.020906,33.697255,61.819279
4,2.22200,HSD17B13,16,-0.408824,27.046854,87.950000,64.169006
5,0.08200,HSD17B13,16,-0.358824,38.983827,63.806250,31.785786
6,0.74100,HSD17B13,16,-0.344118,29.747748,85.943750,59.379379
7,6.66700,HSD17B13,15,-0.264286,26.921848,87.606667,64.596146
8,20.00000,AGT,20,-0.262345,17.170539,77.305000,77.939156
9,0.00064,HSD17B13,13,-0.250000,12.149879,11.537692,12.649751


## Concentration + Patent Hotspots

In [11]:
spearman_by_concentration_patent.sort_values(["spearman", "mae"], ascending=[True, False]).head(20)

,Concentration_nM,patent_group,n_samples,spearman,mae,mean_true,mean_pred
0,10.0000,WO2023213284A1,12,-0.791595,25.177389,84.666667,59.489277
1,15.0000,WO2023091644A2,35,-0.725547,18.390196,86.038000,67.772148
2,3.0000,WO2023091644A2,35,-0.639757,12.128213,71.316857,64.966675
3,1.0000,WO2023213284A1,12,-0.573427,25.973465,76.116667,55.476604
4,10.0000,CN112313335A,182,-0.469431,54.133396,92.877308,38.743912
5,0.1000,CN112313335A,184,-0.467330,87.533814,98.657065,11.123251
6,0.2470,CN116940681A,16,-0.459161,34.587832,78.275000,49.947678
7,2.2220,CN116940681A,16,-0.408824,27.046854,87.950000,64.169006
8,1.0000,CN117448322A,20,-0.389767,51.251217,47.430000,28.507071
9,0.0820,CN116940681A,16,-0.358824,38.983827,63.806250,31.785786


## Experimental Feature Issues

In [12]:
issue_by_cell_type = issue_summary(predictions_df, "Cell_Type", min_samples=30)
issue_by_concentration = issue_summary(predictions_df, "Concentration_nM", min_samples=20)
issue_by_time = issue_summary(predictions_df, "Time_of_administration_h", min_samples=20)
issue_by_patent = issue_summary(predictions_df, "patent_group", min_samples=20)
issue_by_authorization = issue_summary(predictions_df, "Authorization_status", min_samples=20)

issue_by_cell_type.sort_values(["spearman", "mae"], ascending=[True, False]).head(20)

,Cell_Type,n_samples,mae,rmse,spearman,mean_true,mean_pred
0,Non-human hepatocytes,95,35.018343,40.648990,-0.016056,17.390526,52.248890
1,Primary human hepatocytes,2841,25.152228,31.635702,0.104623,51.065164,55.130291
2,Human iPSC-derived cortical neurons,200,25.791386,32.533438,0.269513,42.580000,44.403564
3,Primary Cynomolgus Monkey Hepatocytes,4134,24.779401,31.837367,0.332602,33.570568,37.745693
4,HepG2,1532,23.516439,30.912170,0.358372,39.751469,53.013283
5,Hep3B,11791,28.037398,35.155762,0.382509,42.256907,38.501831
6,Huh7,1545,22.662574,28.178129,0.537952,40.988848,32.955757
7,HEK293 Cells,140,18.350279,23.632800,0.583214,67.108702,66.746819
8,H1299 Cells,1464,12.106355,15.200595,0.607084,68.545902,68.336182
9,COS7,3752,18.931234,23.896948,0.621259,24.735989,31.795238


## Concentration Issue Summary

In [13]:
issue_by_concentration.sort_values(["spearman", "mae"], ascending=[True, False]).head(20)

,Concentration_nM,n_samples,mae,rmse,spearman,mean_true,mean_pred
0,15.00000,35,18.390196,20.182526,-0.725547,86.038000,67.772148
1,3.00000,35,12.128213,14.571704,-0.639757,71.316857,64.966675
2,150.00000,88,30.380491,38.272317,-0.053900,37.002500,63.213364
3,6.66670,48,16.651001,18.916153,-0.011808,78.221042,74.699287
4,0.00030,40,11.501416,14.087672,-0.004128,5.699000,-0.832109
5,0.20000,120,38.182021,44.423139,0.017644,18.021167,10.813755
6,0.00300,64,16.270998,19.496930,0.032169,4.242813,-3.048302
7,8.00000,44,18.810726,23.213902,0.053700,25.563636,42.355930
8,0.00320,111,14.333037,17.386221,0.079968,1.807117,7.212139
9,0.00100,66,17.804429,21.997375,0.102736,7.866212,-3.533337


## Time Issue Summary

In [14]:
issue_by_time.sort_values(["spearman", "mae"], ascending=[True, False]).head(20)

,Time_of_administration_h,n_samples,mae,rmse,spearman,mean_true,mean_pred
0,168.0,200,25.791386,32.533438,0.269513,42.580000,44.403564
1,40.0,194,17.022854,21.932731,0.335873,27.302577,18.885498
2,24.0,23302,25.650162,32.596887,0.439227,41.769514,41.077545
3,72.0,1392,18.530222,23.560465,0.462584,15.625697,26.718067
4,48.0,7865,18.802789,24.415657,0.572890,48.599585,48.273411


## Patent Issue Summary

In [15]:
issue_by_patent.sort_values(["spearman", "mae"], ascending=[True, False]).head(20)

,patent_group,n_samples,mae,rmse,spearman,mean_true,mean_pred
0,CN112313335A,366,70.924863,73.554774,-0.800578,95.782978,24.858116
1,WO2023213284A1,47,30.079176,32.393816,-0.112589,86.337021,57.619560
2,WO2023091644A2,1439,26.844205,33.868907,-0.081932,48.125962,58.125641
3,CN116801886A,272,35.273139,40.432001,-0.025793,50.522426,39.465919
4,TW202321444A,95,35.018343,40.648990,-0.016056,17.390526,52.248890
5,CN117210468A,222,21.069589,26.999953,0.044493,25.183333,24.021629
6,WO2022121959A1,95,36.469945,43.809732,0.091092,65.644737,52.002277
7,CN117448322A,132,46.753006,52.871900,0.107533,45.159848,11.498766
8,WO2023056446A1,223,22.308039,27.449982,0.127449,61.098655,54.035007
9,WO2022089486A1,253,15.899085,20.136795,0.191188,78.988142,68.469597


## Authorization Issue Summary

In [16]:
issue_by_authorization.sort_values(["spearman", "mae"], ascending=[True, False]).head(20)

,Authorization_status,n_samples,mae,rmse,spearman,mean_true,mean_pred
0,Published,222,21.069589,26.999953,0.044493,25.183333,24.021629
1,Substantive Examination,15513,26.294885,33.076359,0.484623,45.976390,36.954094
2,Withdrawn,1080,23.935436,29.820043,0.503063,30.217500,24.032166
3,Unknown Status,12202,22.657121,29.420326,0.532386,33.724793,45.597641
4,Granted,1619,21.655140,27.752175,0.602926,49.050797,47.594902
5,UNKNOWN,2333,12.920672,16.425489,0.661400,63.958475,63.915451


## Prediction Preview

In [17]:
predictions_df[["group", "Cell_Type", "Concentration_nM", "Time_of_administration_h", "patent_group", "y_true", "y_pred", "residual", "abs_error"]].head(20)

,group,Cell_Type,Concentration_nM,Time_of_administration_h,patent_group,y_true,y_pred,residual,abs_error
0,CTNNB1,Hela,100.0,48.0,CN107365771B,88.00,75.181297,12.818703,12.818703
1,CTNNB1,Hela,100.0,48.0,CN107365771B,90.00,99.503929,-9.503929,9.503929
2,CTNNB1,Hela,100.0,48.0,CN107365771B,90.00,67.125229,22.874771,22.874771
3,CTNNB1,Hela,100.0,48.0,CN107365771B,89.00,88.503868,0.496132,0.496132
4,CTNNB1,Hela,100.0,48.0,CN107365771B,87.00,61.742676,25.257324,25.257324
5,CTNNB1,Hela,100.0,48.0,CN107365771B,91.00,117.396027,-26.396027,26.396027
6,CTNNB1,Hela,100.0,48.0,CN107365771B,62.00,71.198021,-9.198021,9.198021
7,CTNNB1,Hep3B,10.0,24.0,WO2023003995A8,93.62,117.932846,-24.312846,24.312846
8,CTNNB1,Hep3B,10.0,24.0,WO2023003995A8,92.55,94.901222,-2.351222,2.351222
9,CTNNB1,Hep3B,10.0,24.0,WO2023003995A8,93.48,96.807961,-3.327961,3.327961


## Interpretation Notes

- This final focused XGBoost ablation is the strongest XGBoost cleanup result so far.
- Overall metrics improve from the original by-gene baseline of `Spearman 0.4589`, `Pearson 0.4534`, `RMSE 31.42`, and `MAE 24.57` to `Spearman 0.4936`, `Pearson 0.4914`, `RMSE 30.43`, and `MAE 23.66`.
- That is a meaningful gain across all four top-level metrics, and it is stronger than both the earlier single patent-hotspot ablation and the earlier three-gene-hotspot ablation.
- This means the five removed subsets were not just ordinary hard examples. Together they were genuinely hurting transfer to unseen genes, which is exactly the behavior we care about before deep learning.
- The result supports keeping this 5-hotspot filter as the current best XGBoost-based cleaned dataset version.
- At the same time, the post-ablation hotspot tables show that the problem is not completely solved. The remaining `Concentration + Gene` failures are now dominated by `HSD17B13` across several concentrations, especially `15 nM`, `3 nM`, `0.247`, `5`, `2.222`, `0.082`, and `0.741`.
- On the patent side, `CN112313335A` remains the clearest unresolved warning sign. It still has extremely poor patent-level behavior, and its `10 nM` and `0.1 nM` concentration slices remain among the weakest rows. `WO2023091644A2` also continues to appear repeatedly in the remaining worst patent combinations.
- Cell-type behavior improves relative to the original by-gene notebook because `Primary mouse hepatocytes` was removed, but `Non-human hepatocytes` still looks weak and some broader hepatocyte contexts are still only moderately learnable.
- Time-of-administration still does not look like the main poisoning axis here. The notebook does not support broad deletion of all `72h` rows.
- So the right interpretation is: this focused ablation materially improves the data for XGBoost, but some residual target-specific and patent-specific transfer problems remain.

## Conclusions And Next Step

- This 5-hotspot ablation is probably the strongest XGBoost-based filtered dataset version tested so far.
- It is strong enough to be treated as a serious candidate pre-CNN cleanup set.
- The main remaining residual-risk areas are:
  - `HSD17B13` concentration hotspots
  - `CN112313335A` patent hotspots
  - repeated `WO2023091644A2` weak slices
- I would not keep iterating forever. This result is already meaningful and likely good enough to move toward deep learning if we document the remaining risk areas clearly.
- The best next step is to compare this final XGBoost-cleaned version against the best RF-cleaned version and decide whether to:
  - stop here and use this cleaned data for CNN training, or
  - run one last very small targeted ablation around `HSD17B13` or `CN112313335A` only.
- My recommendation is to treat this notebook as the current XGBoost winner and only do one more ablation if you feel you still need extra confidence before CNN training.
